In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

import datetime as dt
import talib as ta

In [2]:
# train & test period
train_start_date = "2018-01-01"
train_end_date = "2021-12-31"

test_start_date = "2022-01-01"
test_end_date = "2026-01-01"

In [3]:
# whole market data

market_ticker = yf.Ticker("SPY")

market_data = market_ticker.history(
    start=train_start_date,
    end=test_end_date
)

In [4]:
market_data


,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains
Date,,,,,,,,
2018-01-02 00:00:00-05:00,235.137825,235.989393,234.751545,235.954269,86655700,0.0,0.0,0.0
2018-01-03 00:00:00-05:00,236.121088,237.595987,236.121088,237.446732,90070400,0.0,0.0,0.0
2018-01-04 00:00:00-05:00,238.087577,238.930358,237.508158,238.447495,80636400,0.0,0.0,0.0
2018-01-05 00:00:00-05:00,239.237679,240.159468,238.746055,240.036575,83524000,0.0,0.0,0.0
2018-01-08 00:00:00-05:00,239.939959,240.633511,239.650262,240.475494,57319200,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
2025-12-24 00:00:00-05:00,684.313382,687.178163,684.164151,686.730530,39445600,0.0,0.0,0.0
2025-12-26 00:00:00-05:00,686.989161,688.003728,685.626408,686.660889,41613300,0.0,0.0,0.0
2025-12-29 00:00:00-05:00,683.905508,685.556767,682.443308,684.213867,62559500,0.0,0.0,0.0


In [5]:
# convert to return
market_data["Market Return"] = market_data["Close"].pct_change()


# convert to sd on 20-day 
market_data["Rolling Volatility"] = (
    market_data["Market Return"]
    .rolling(window=20)
    .std()
)

In [6]:
market_data["Rolling Volatility"]

Date
2018-01-02 00:00:00-05:00         NaN
2018-01-03 00:00:00-05:00         NaN
2018-01-04 00:00:00-05:00         NaN
2018-01-05 00:00:00-05:00         NaN
2018-01-08 00:00:00-05:00         NaN
                               ...   
2025-12-24 00:00:00-05:00    0.005627
2025-12-26 00:00:00-05:00    0.005475
2025-12-29 00:00:00-05:00    0.005454
2025-12-30 00:00:00-05:00    0.005341
2025-12-31 00:00:00-05:00    0.005626
Name: Rolling Volatility, Length: 2011, dtype: float64

In [7]:
market_data["Market Return"]

Date
2018-01-02 00:00:00-05:00         NaN
2018-01-03 00:00:00-05:00    0.006325
2018-01-04 00:00:00-05:00    0.004215
2018-01-05 00:00:00-05:00    0.006664
2018-01-08 00:00:00-05:00    0.001829
                               ...   
2025-12-24 00:00:00-05:00    0.003518
2025-12-26 00:00:00-05:00   -0.000101
2025-12-29 00:00:00-05:00   -0.003564
2025-12-30 00:00:00-05:00   -0.001221
2025-12-31 00:00:00-05:00   -0.007409
Name: Market Return, Length: 2011, dtype: float64

In [8]:
# train & test data

train_data = market_data.loc[
    train_start_date:train_end_date
].copy()

test_data = market_data.loc[
    test_start_date:test_end_date
].copy()

In [9]:
# use traindata median for now

median_volatility = train_data["Rolling Volatility"].median()


In [10]:
median_volatility

0.00844931179515919

In [13]:
market_data["Period"] = np.where(
    market_data.index <= train_end_date,
    "Train",
    "Test"
)

In [14]:
market_data["Market Regime"] = np.where(
    market_data["Rolling Volatility"] > median_volatility,
    "High Volatility",
    "Low Volatility"
)

In [15]:
market_regime = market_data[
    [
        "Market Return",
        "Rolling Volatility",
        "Market Regime",
        "Period"
    ]
]

market_regime

,Market Return,Rolling Volatility,Market Regime,Period
Date,,,,
2018-01-02 00:00:00-05:00,NaN,NaN,Low Volatility,Train
2018-01-03 00:00:00-05:00,0.006325,NaN,Low Volatility,Train
2018-01-04 00:00:00-05:00,0.004215,NaN,Low Volatility,Train
2018-01-05 00:00:00-05:00,0.006664,NaN,Low Volatility,Train
2018-01-08 00:00:00-05:00,0.001829,NaN,Low Volatility,Train
...,...,...,...,...
2025-12-24 00:00:00-05:00,0.003518,0.005627,Low Volatility,Test
2025-12-26 00:00:00-05:00,-0.000101,0.005475,Low Volatility,Test
2025-12-29 00:00:00-05:00,-0.003564,0.005454,Low Volatility,Test


In [16]:
market_regime.to_csv(
    "market_regime_split.csv")